In [1]:
!pip install -q sentence-transformers faiss-cpu gradio transformers beautifulsoup4

In [2]:
import re
import json
import urllib.request
from html import unescape
from bs4 import BeautifulSoup

def fetch_html(url):
    headers = {
        "User-Agent": "Mozilla/5.0"
    }
    req = urllib.request.Request(url, headers=headers)
    with urllib.request.urlopen(req) as response:
        return response.read().decode('utf-8')

def extract_menu_json(html_content):
    script_blocks = re.findall(r'<script[^>]+type="application/ld\+json"[^>]*>\s*(\{.*?\})\s*</script>', html_content, re.DOTALL)
    for block in script_blocks:
        try:
            data = json.loads(block)
            if data.get("@type") == "Menu":
                return data
        except json.JSONDecodeError:
            continue
    return None

def parse_item_names_from_html(soup):
    item_names = []
    all_h4_tags = soup.find_all("h4")
    class_lengths = [len(h4.get("class", [])) for h4 in all_h4_tags if h4.get("class")]
    if not class_lengths:
        return item_names
    section_class_len = max(class_lengths)
    item_class_len = min(class_lengths)
    for h4_tag in all_h4_tags:
        text = h4_tag.get_text(strip=True)
        classes = h4_tag.get("class", [])
        if not classes:
            continue
        if len(classes) == item_class_len:
            item_names.append(text)
    return item_names

def extract_menu_items(menu_json, fallback_item_names):
    menu_items = []
    item_counter = 1
    fallback_idx = 0
    for section in menu_json.get("hasMenuSection", []):
        section_name = section.get("name", "Unnamed Section")
        for subsection in section.get("hasMenuSection", []):
            for item in subsection.get("hasMenuItem", []):
                name = item.get("name", "").strip()
                description = item.get("description", "").strip()
                price = item.get("offers", {}).get("price")
                if not name:
                    if fallback_idx < len(fallback_item_names):
                        name = fallback_item_names[fallback_idx]
                        fallback_idx += 1
                    elif description:
                        name = description.split("(")[0].strip()
                    else:
                        name = f"Unnamed Item #{item_counter}"
                        item_counter += 1
                menu_items.append({
                    "section": section_name,
                    "item": unescape(name),
                    "price": price
                })
    return menu_items

def scrape_restaurant(url):
    html_content = fetch_html(url)
    soup = BeautifulSoup(html_content, "html.parser")
    menu_json = extract_menu_json(html_content)
    if not menu_json:
        print(f"Menu JSON not found for {url}. Skipping...")
        return []
    fallback_item_names = parse_item_names_from_html(soup)
    menu_items = extract_menu_items(menu_json, fallback_item_names)
    return menu_items

urls = [
    "https://www.zomato.com/ncr/sita-ram-diwan-chand-paharganj-new-delhi/order",
    "https://www.zomato.com/ncr/gulati-pandara-road-new-delhi/order",
    "https://www.zomato.com/ncr/karims-jama-masjid-jama-masjid-new-delhi/order",
    "https://www.zomato.com/ncr/havemore-pandara-road-new-delhi/order",
    "https://www.zomato.com/ncr/qureshis-kabab-corner-defence-colony-new-delhi/order"
]

all_data = {}

for url in urls:
    print(f"Scraping {url}")
    name = url.split("/")[-2]
    all_data[name] = scrape_restaurant(url)

with open("scraped_menus.json", "w", encoding="utf-8") as f:
    json.dump(all_data, f, ensure_ascii=False, indent=2)

print("Scraping complete.")


Scraping https://www.zomato.com/ncr/sita-ram-diwan-chand-paharganj-new-delhi/order
Scraping https://www.zomato.com/ncr/gulati-pandara-road-new-delhi/order
Scraping https://www.zomato.com/ncr/karims-jama-masjid-jama-masjid-new-delhi/order
Scraping https://www.zomato.com/ncr/havemore-pandara-road-new-delhi/order
Scraping https://www.zomato.com/ncr/qureshis-kabab-corner-defence-colony-new-delhi/order
Scraping complete.


In [3]:
from sentence_transformers import SentenceTransformer
import faiss

with open("scraped_menus.json", "r", encoding="utf-8") as f:
    restaurant_data = json.load(f)

embedder = SentenceTransformer('all-MiniLM-L6-v2')

texts, metadata = [], []

for restaurant, items in restaurant_data.items():
    for item in items:
        section = item.get("section", "")
        name = item.get("item", "")
        price = item.get("price", "")
        text = f"Restaurant: {restaurant}. Section: {section}. Item: {name}. Price: ₹{price}"
        texts.append(text)
        metadata.append({
            "restaurant": restaurant,
            "section": section,
            "item": name,
            "price": price
        })

embeddings = embedder.encode(texts, convert_to_numpy=True)
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print(f"FAISS index created with {len(texts)} items.")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


FAISS index created with 583 items.


In [4]:
from transformers import pipeline

# Load the flan-t5-large model
generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-large",
    max_new_tokens=256,
    temperature=0.7,
    top_p=0.9
)

Device set to use cuda:0


In [5]:
def rag_chatbot(query):
    query_embedding = embedder.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_embedding, k=3)
    retrieved_texts = [texts[i] for i in indices[0] if i < len(texts)]

    if not retrieved_texts or distances[0][0] > 80:
        return "Sorry, I couldn't find relevant information."

    context = "\n".join(retrieved_texts)

    # Simple and effective prompt
    prompt = f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"

    # Generate response
    response = generator(prompt)[0]['generated_text'].strip()

    # Optional trimming
    return response.split("\n")[0] if response else "Sorry, no answer found."


In [ ]:
import gradio as gr

chat_history = []

def chatbot_interface(user_query):
    global chat_history
    response = rag_chatbot(user_query)
    chat_history.append(("User", user_query))
    chat_history.append(("Bot", response))
    return chat_history

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("## 🍽️ **Zomato Menu Chatbot**")
    gr.Markdown("Ask about menu items, prices, vegetarian options, desserts, and more!\n")

    chatbot = gr.Chatbot(height=400)
    with gr.Row():
        user_input = gr.Textbox(placeholder="Type your question here...", show_label=False, lines=1, container=False)
        send_btn = gr.Button("Send")

    def handle_submit(message):
        global chat_history
        response = rag_chatbot(message)
        chat_history.append(("User", message))
        chat_history.append(("Bot", response))
        return "", chat_history

    send_btn.click(handle_submit, inputs=user_input, outputs=[user_input, chatbot])
    user_input.submit(handle_submit, inputs=user_input, outputs=[user_input, chatbot])  # Pressing Enter also works

demo.launch(debug=True, share=True)


<ipython-input-6-d4f387d97157>:16: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(height=400)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://5359c286e49e31c04e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
